# Colab experiment: tracing mathematical error detection

This single notebook contains the complete preregistered ProcessBench workflow with `Qwen/Qwen2.5-Math-1.5B-Instruct`: environment tests, a smoke/full run switch, Drive persistence, activation extraction, probes and controls, cross-domain transfer, PCA accessibility, causal interventions, result inspection, and paper-ready plots. Start with `RUN_MODE = "smoke"`, then change it to `"full"`. Use a T4 GPU.

## 1. GitHub authentication, repository, and dependencies

Create a fine-grained GitHub personal access token with **Contents: read and write** access to this repository. Store it as a private Colab secret named `GITHUB_TOKEN` (recommended), set it as an environment variable, or enter it in the hidden prompt. The token is passed through a temporary `GIT_ASKPASS` environment: it is never placed in the clone URL, Git remote, notebook source, or cell output.

In [ ]:
import os
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path


def load_github_token() -> str:
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        try:
            from google.colab import userdata

            token = (userdata.get("GITHUB_TOKEN") or "").strip()
        except Exception:
            token = ""
    if not token:
        token = getpass("GitHub token (hidden): ").strip()
    if not token:
        raise RuntimeError("A GitHub token is required to clone and publish results.")
    return token


GITHUB_TOKEN = load_github_token()
askpass_path = Path(tempfile.gettempdir()) / "math_error_github_askpass.sh"
askpass_path.write_text(
    (
        "#!/bin/sh\n"
        'case "$1" in\n'
        "*Username*) echo x-access-token ;;\n"
        '*Password*) printf "%s\n" "$GITHUB_TOKEN" ;;\n'
        "esac\n"
    ),
    encoding="utf-8",
)
askpass_path.chmod(0o700)


def github_git_env() -> dict[str, str]:
    environment = os.environ.copy()
    environment.update(
        {
            "GITHUB_TOKEN": GITHUB_TOKEN,
            "GIT_ASKPASS": str(askpass_path),
            "GIT_ASKPASS_REQUIRE": "force",
            "GIT_TERMINAL_PROMPT": "0",
        }
    )
    return environment


REPO_URL = (
    "https://github.com/sagnikc395/tracing-mathematical-error-detection-in-language-models.git"
)
AUTHENTICATED_REPO_URL = REPO_URL.replace("https://", "https://x-access-token@", 1)
REPO_NAME = "tracing-mathematical-error-detection-in-language-models"
cwd = Path.cwd()

if (cwd / "pyproject.toml").exists():
    repo_dir = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_dir = cwd.parent
else:
    repo_dir = Path("/content") / REPO_NAME
    if not (repo_dir / "pyproject.toml").exists():
        subprocess.run(
            ["git", "clone", AUTHENTICATED_REPO_URL, str(repo_dir)],
            check=True,
            env=github_git_env(),
        )

os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
print(f"Repository: {repo_dir}")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. In Colab select Runtime → Change runtime type → T4 GPU.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}")
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 2. Choose smoke or full mode and configure storage

Smoke mode uses 25 traces per source, disables bootstrap resampling, and uses smaller causal controls. Full mode restores every preregistered setting and persists to Drive by default because Colab runtimes expire. The modes always use separate data and artifact paths.

In [ ]:
RUN_MODE = "smoke"  # Change to "full" after the smoke run succeeds.
USE_GOOGLE_DRIVE = RUN_MODE == "full"
DRIVE_PROJECT_DIR = "math-error-tracing"

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError('RUN_MODE must be either "smoke" or "full"')

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as error:
        raise RuntimeError(
            "Drive mounting requires Colab; set USE_GOOGLE_DRIVE = False locally."
        ) from error
    drive.mount("/content/drive")
    run_root = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
else:
    run_root = repo_dir

data_name = "processbench-smoke.jsonl" if RUN_MODE == "smoke" else "processbench.jsonl"
artifact_name = "smoke" if RUN_MODE == "smoke" else "qwen2.5-math-1.5b"
data_path = run_root / "data" / data_name
artifact_dir = run_root / "artifacts" / artifact_name
data_path.parent.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
print(f"Run mode: {RUN_MODE}")
print(f"Run root: {run_root}")

In [ ]:
import tempfile

import yaml

with open("configs/experiment.yaml", encoding="utf-8") as handle:
    experiment_config = yaml.safe_load(handle)

experiment_config["data"]["output_path"] = str(data_path)
experiment_config["extraction"]["output_dir"] = str(artifact_dir)
if RUN_MODE == "smoke":
    experiment_config["data"]["max_examples_per_split"] = 25
    experiment_config["probe"]["bootstrap_samples"] = 0
    experiment_config["intervention"]["examples_per_class"] = 8
    experiment_config["intervention"]["random_directions"] = 2
else:
    experiment_config["data"]["max_examples_per_split"] = None

session_dir = Path("/content") if Path("/content").exists() else Path(tempfile.gettempdir())
config_path = session_dir / f"math_error_{RUN_MODE}.yaml"
config_path.write_text(yaml.safe_dump(experiment_config, sort_keys=False), encoding="utf-8")
print(config_path.read_text())

## 3. Select stages

Each stage is independent and resumable. Leave completed stages enabled if desired: data download returns immediately, complete activation shards are skipped, and later artifacts are safely regenerated. Set the intervention flag to `False` when you want probe results first and plan to return for the slow causal run.

In [ ]:
RUN_DOWNLOAD = True
RUN_EXTRACTION = True
RUN_PROBES_AND_CONTROLS = True
RUN_INTERVENTIONS = RUN_MODE == "full"  # Set True to include the causal smoke test.
RUN_PLOTS = True


def run_stage(stage: str, *extra: str) -> None:
    command = [sys.executable, "-m", "causal_circuits", "--config", str(config_path), stage, *extra]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)


run_stage("validate-config")

## 4. Download ProcessBench

In [ ]:
if RUN_DOWNLOAD:
    run_stage("download-data")
else:
    print("Download skipped.")

## 5. Extract step-boundary activations

The complete prompt is forwarded once per trace and every step boundary is saved. Traces over 2,048 tokens are logged and excluded, never truncated. Completed 100-trace shards are reused after a disconnect.

In [ ]:
if RUN_EXTRACTION:
    run_stage("extract-activations")
else:
    print("Extraction skipped.")

In [ ]:
import json

import pandas as pd
from IPython.display import display

manifest_paths = sorted((artifact_dir / "activation_shards").glob("shard_*.json"))
manifest_rows = [json.loads(path.read_text()) for path in manifest_paths]
if manifest_rows:
    manifests = pd.DataFrame(manifest_rows)
    display(manifests.drop(columns=["skipped"], errors="ignore").sum().to_frame("total").T)
    print(f"Complete activation shards: {len(manifest_paths)}")
else:
    print("No shard manifests found yet.")

## 6. Fit probes, controls, transfer matrix, and PCA curve

This stage selects hyperparameters and layers on validation data, then evaluates the held-out test set. Full mode uses 1,000 trace-grouped bootstrap samples; smoke mode skips the bootstrap for speed.

In [ ]:
if RUN_PROBES_AND_CONTROLS:
    run_stage("fit-probes")
else:
    print("Probe stage skipped.")

## 7. Inspect the predictive and localization results

In [ ]:
import numpy as np

probe_dir = artifact_dir / "probes"
direction_artifact = np.load(probe_dir / "directions.npz")
selected_layer = int(direction_artifact["selected_layer"])
selected_intervention_layer = int(direction_artifact["selected_intervention_layer"])
metrics = pd.read_csv(probe_dir / "layer_metrics.csv")
controls = pd.read_csv(probe_dir / "controls.csv")
transfer = pd.read_csv(probe_dir / "domain_transfer.csv")
pca_curve = pd.read_csv(probe_dir / "pca_subspace.csv")
bootstrap_path = probe_dir / "test_group_bootstrap.csv"
bootstrap = pd.read_csv(bootstrap_path) if bootstrap_path.stat().st_size > 1 else pd.DataFrame()

print(f"Validation-selected probe layer: {selected_layer}")
print(f"Validation-selected intervention layer: {selected_intervention_layer}")
display(
    metrics[
        (metrics.layer == selected_layer)
        & metrics.split.isin(["validation", "test", "test_error_traces"])
    ]
)
display(controls)

In [ ]:
if not bootstrap.empty:
    interval_columns = ["auroc", "average_precision", "process_f1", "first_error_exact"]
    intervals = bootstrap[interval_columns].quantile([0.025, 0.5, 0.975]).T
    intervals.columns = ["2.5%", "median", "97.5%"]
    print("Trace-grouped bootstrap intervals")
    display(intervals)
else:
    print("Bootstrap intervals are disabled in smoke mode.")

print("Cross-domain AUROC (train source × test source)")
display(transfer.pivot(index="train_source", columns="test_source", values="auroc"))
print("Cross-domain first-error F1")
display(transfer.pivot(index="train_source", columns="test_source", values="process_f1"))

print("Top-variance subspace accessibility")
display(pca_curve)

## 8. Run held-out causal interventions

This is the slowest stage. It measures a seven-point signed dose response for the learned direction and compares the extremes with 20 matched random orthogonal directions. Outputs are written incrementally only when the stage completes, so budget a stable runtime.

In [ ]:
if RUN_INTERVENTIONS:
    run_stage("run-interventions")
else:
    print("Interventions skipped. Set RUN_INTERVENTIONS = True above when ready.")

In [ ]:
intervention_dir = artifact_dir / "interventions"
summary_path = intervention_dir / "summary.csv"
if summary_path.exists():
    intervention_summary = pd.read_csv(summary_path)
    behavioral = json.loads((intervention_dir / "behavioral_verdict.json").read_text())
    print("Unmodified behavioral verdict metrics:", behavioral)
    display(intervention_summary[intervention_summary.direction_type == "learned"])
else:
    print("No intervention summary found.")

## 9. Generate and inspect figures

In [ ]:
if RUN_PLOTS:
    run_stage("plot")
else:
    print("Plotting skipped.")

In [ ]:
import matplotlib.pyplot as plt

test_metrics = metrics[metrics["split"] == "test"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(test_metrics["layer"], test_metrics["auroc"], marker="o", label="step AUROC")
axes[0].plot(test_metrics["layer"], test_metrics["process_f1"], marker="s", label="first-error F1")
axes[0].axhline(0.5, color="0.6", linestyle="--", linewidth=1)
axes[0].set(
    xlabel="Hidden-state index", ylabel="Held-out score", ylim=(0, 1), title="Layer-wise probe"
)
axes[0].legend(frameon=False)

auroc_matrix = transfer.pivot(index="train_source", columns="test_source", values="auroc")
image = axes[1].imshow(auroc_matrix, vmin=0, vmax=1, cmap="viridis")
axes[1].set_xticks(range(len(auroc_matrix.columns)), auroc_matrix.columns, rotation=35, ha="right")
axes[1].set_yticks(range(len(auroc_matrix.index)), auroc_matrix.index)
axes[1].set(xlabel="Test source", ylabel="Train source", title="Cross-domain AUROC")
fig.colorbar(image, ax=axes[1], label="AUROC")
fig.tight_layout()
plt.show()

if summary_path.exists():
    learned = intervention_summary[intervention_summary.direction_type == "learned"]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(learned["alpha"], learned["mean_delta"], yerr=learned["standard_error"], marker="o")
    ax.axhline(0, color="0.6", linewidth=1)
    ax.set(
        xlabel=r"Intervention strength $\alpha$ (projection SD)",
        ylabel="Change in verdict score",
        title="Learned-direction dose response",
    )
    plt.show()

## Completion checklist

Before writing the result narrative, confirm that the artifact directory contains layer metrics, controls, both transfer metrics, trace-grouped bootstrap samples, intervention baselines and individual rows, and the three PDF figures. Interpret the result using the preregistered decision table in `experiment.md`; decodability by itself is not causal evidence.

In [ ]:
expected = [
    "probes/layer_metrics.csv",
    "probes/controls.csv",
    "probes/domain_transfer.csv",
    "probes/test_group_bootstrap.csv",
    "probes/directions.npz",
    "figures/layerwise_probe.pdf",
    "figures/pca_subspace.pdf",
]
if RUN_INTERVENTIONS:
    expected += [
        "interventions/individual.csv",
        "interventions/summary.csv",
        "interventions/behavioral_verdict.json",
        "figures/causal_dose_response.pdf",
    ]

checklist = pd.DataFrame(
    {"artifact": expected, "exists": [(artifact_dir / item).exists() for item in expected]}
)
display(checklist)
if not checklist["exists"].all():
    print("Some requested artifacts are missing; rerun the corresponding stage before analysis.")
else:
    print(f"{RUN_MODE.title()} result package is complete under {artifact_dir}")

## 10. Publish final results back to GitHub

Run this cell only after the completion checklist passes. It copies the enabled run's final outputs into `artifacts/<run-name>/`, records the exact generated config, force-adds the otherwise ignored result paths, commits, rebases on the current remote branch, and pushes using the same token. Activation shards are excluded by default because they are resumable caches and can exceed GitHub's file/repository limits. The size guard rejects files above 95 MiB; use a separate Git LFS-aware workflow if you need to publish those caches. Dataset files and the token are never committed.

In [ ]:
import shutil

PUBLISH_TO_GITHUB = True
PUBLISH_ACTIVATION_SHARDS = False
GITHUB_BRANCH = "main"
GIT_AUTHOR_NAME = "Colab Experiment Runner"
GIT_AUTHOR_EMAIL = "colab-experiments@users.noreply.github.com"
MAX_REGULAR_GITHUB_FILE_BYTES = 95 * 1024 * 1024

if PUBLISH_TO_GITHUB:
    if not checklist["exists"].all():
        raise RuntimeError("The result checklist is incomplete; publishing was stopped.")

    staged_before = subprocess.run(
        ["git", "diff", "--cached", "--name-only"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if staged_before:
        raise RuntimeError(
            "The repository already has staged changes; commit or unstage them first."
        )

    publish_dir = repo_dir / "artifacts" / artifact_name
    if artifact_dir.resolve() != publish_dir.resolve():
        ignored = (
            shutil.ignore_patterns("activation_shards") if not PUBLISH_ACTIVATION_SHARDS else None
        )
        shutil.copytree(artifact_dir, publish_dir, dirs_exist_ok=True, ignore=ignored)
    publish_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(config_path, publish_dir / "experiment_config.yaml")

    publish_items = ["extraction_identity.json", "experiment_config.yaml", "probes", "figures"]
    if (publish_dir / "interventions").exists():
        publish_items.append("interventions")
    if PUBLISH_ACTIVATION_SHARDS:
        publish_items.append("activation_shards")

    publish_paths = [publish_dir / item for item in publish_items if (publish_dir / item).exists()]
    oversized = [
        path
        for root in publish_paths
        for path in ([root] if root.is_file() else root.rglob("*"))
        if path.is_file() and path.stat().st_size > MAX_REGULAR_GITHUB_FILE_BYTES
    ]
    if oversized:
        names = ", ".join(str(path.relative_to(repo_dir)) for path in oversized)
        raise RuntimeError(
            f"Files exceed the safe GitHub limit: {names}. Use Git LFS or exclude them."
        )

    relative_paths = [str(path.relative_to(repo_dir)) for path in publish_paths]
    subprocess.run(["git", "add", "-f", "--", *relative_paths], check=True)
    staged = subprocess.run(
        ["git", "diff", "--cached", "--name-only", "--", *relative_paths],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()

    if not staged:
        print("Published artifacts are already up to date; nothing to commit.")
    else:
        subprocess.run(["git", "config", "user.name", GIT_AUTHOR_NAME], check=True)
        subprocess.run(["git", "config", "user.email", GIT_AUTHOR_EMAIL], check=True)
        commit_message = f"Add {RUN_MODE} experiment artifacts"
        subprocess.run(["git", "commit", "-m", commit_message, "--", *relative_paths], check=True)
        subprocess.run(
            ["git", "pull", "--rebase", AUTHENTICATED_REPO_URL, GITHUB_BRANCH],
            check=True,
            env=github_git_env(),
        )
        subprocess.run(
            ["git", "push", AUTHENTICATED_REPO_URL, f"HEAD:{GITHUB_BRANCH}"],
            check=True,
            env=github_git_env(),
        )
        print(f"Published {len(relative_paths)} result paths to origin/{GITHUB_BRANCH}.")
else:
    print("GitHub publishing disabled.")